# LMA Phase 3: TELUGU Reasoning Finetuning -- LOW-PARAMETER ABLATION

Finetunes the Phase 2 **submission** checkpoint (6 layers, 10K WordPiece vocab, 7.34M params, PPL 881.9 -- `telugu/model/outputs/submission/checkpoint_best.pt`) on the same synthetic comparative-reasoning QA set used by the normal `finetune_kaggle.ipynb`, using the same finetuning code from the same bundle (`kspsvln/lma-telugu-phase2`) but a **separate** config (`finetune_low_config.json`) that does not touch `finetune_config.json` or its checkpoints/logs.

This exists to run the low-vs-high-parameter Telugu ablation described in `report/phase-3/report.md` Sec. 3: everything (data, hyperparameters, schedule) is kept identical to the normal run -- only the checkpoint/architecture being finetuned differs -- so the two runs' results are directly comparable.

In [1]:
# ===== CONFIG-ONLY CELL: Modify these before running =====
ROOT_DIR = "/kaggle/input/datasets/kspsvln/lma-telugu-phase2"  # SAME code+config bundle as the normal run -- just re-upload after the finetune_low_config.json / tokenizer_low / finetune.py changes
PRETRAINED_CKPT = "/kaggle/input/datasets/kspsvln/checkpoint-telugu/submission_phase-2/checkpoint_best.pt"  # Phase 2 SUBMISSION checkpoint -- upload telugu/model/outputs/submission/checkpoint_best.pt as a NEW Kaggle dataset (e.g. "checkpoint-telugu-low") and attach it; this is NOT the same checkpoint as the normal notebook uses. This is a best-guess path -- the local source folder is named "submission/" (not "checkpoints/" like the normal notebook's dataset), so if you preserved that folder name on upload the real path will be .../submission/checkpoint_best.pt instead; the next cell auto-detects and corrects this if the literal path above doesn't exist, so you don't have to get it exactly right here.
OUT_DIR = "/kaggle/working/finetune_checkpoints_low"   # Separate output dir -- never overlaps with the normal run's finetune_checkpoints/
CHECK_DIR = None                                    # Resume finetuning from a previous LOW-param finetune session (if available)

LANGUAGE = "telugu"
FINETUNE_CONFIG_NAME = "finetune_low_config.json"  # Separate config -- carries the low-parameter model_architecture + tokenizer overrides, does not touch finetune_config.json

# Hyperparameters: None means use finetune_low_config.json defaults (kept identical to
# finetune_config.json's values on purpose, for a clean ablation -- only architecture differs).
hp = dict(
    batch_size=None,
    learning_rate=None,
    num_epochs=None,
    warmup_steps=None,
    weight_decay=None,
    amp=True,
)

In [2]:
import sys
import os
import argparse
from pathlib import Path

# Fail fast if ROOT_DIR isn't actually where the bundle is mounted, and if the low-parameter
# config/tokenizer files aren't in it yet (they're only present after re-uploading the bundle
# following this ablation's setup -- an older bundle version would be missing them).
root_path = Path(ROOT_DIR)
expected_entry = root_path / "finetune" / "finetune.py"
expected_low_config = root_path / "configs" / FINETUNE_CONFIG_NAME
expected_low_tokenizer = root_path / "tokenizer" / "full_wordPiece_level" / "telugu_wp_tokenizer_low.json"
if not expected_entry.exists():
    kaggle_input = Path("/kaggle/input")
    available = sorted(p.name for p in kaggle_input.iterdir()) if kaggle_input.exists() else []
    listing = "\n".join(f"  {p}" for p in sorted(root_path.iterdir())) if root_path.exists() else "  (ROOT_DIR does not exist)"
    raise RuntimeError(
        f"Expected {expected_entry} but it's not there.\n"
        f"ROOT_DIR = {ROOT_DIR}\n"
        f"Contents of ROOT_DIR:\n{listing}\n"
        f"Datasets attached under /kaggle/input/: {available}\n"
        "Check that the lma-telugu-phase2 bundle is attached as an input to this notebook "
        "(Add Input) and that ROOT_DIR above matches its actual mounted path."
    )
if not expected_low_config.exists() or not expected_low_tokenizer.exists():
    raise RuntimeError(
        f"{expected_low_config} or {expected_low_tokenizer} not found in the attached bundle. "
        "This notebook needs the low-parameter ablation files (finetune_low_config.json, "
        "tokenizer_config_low.json, telugu_wp_tokenizer_low.json) -- re-export/re-upload the "
        "kaggle_bundle after they were added, then re-attach the updated dataset version."
    )

if not Path(PRETRAINED_CKPT).exists():
    # The literal path above is a best guess -- the exact nesting under the attached dataset
    # depends on how it was uploaded (this checkpoint's local source folder is named
    # "submission/", not "checkpoints/" like the normal notebook's checkpoint-telugu dataset,
    # so a hardcoded guess is fragile). Search the dataset root for checkpoint_best.pt instead
    # of assuming a layout.
    input_root = Path('/kaggle/input')
    dataset_root = None
    for parent in Path(PRETRAINED_CKPT).parents:
        if parent.parent == input_root:
            dataset_root = parent
            break
    search_root = dataset_root if dataset_root and dataset_root.exists() else input_root
    matches = sorted(search_root.rglob('checkpoint_best.pt')) if search_root.exists() else []
    if len(matches) == 1:
        print(f'⚠️  PRETRAINED_CKPT={PRETRAINED_CKPT} not found as given -- auto-detected the only '
              f'checkpoint_best.pt under {search_root}: {matches[0]}. Using that instead.')
        PRETRAINED_CKPT = str(matches[0])
    elif len(matches) > 1:
        raise RuntimeError(
            f'PRETRAINED_CKPT={PRETRAINED_CKPT} not found, and multiple checkpoint_best.pt files '
            f'exist under {search_root}:\n' + '\n'.join(f'  {m}' for m in matches) +
            '\nSet PRETRAINED_CKPT above to the correct one explicitly -- it must be the SUBMISSION '
            'checkpoint (6 layers, 10K vocab, ~88MB), not the normal checkpoint-telugu dataset '
            '(25.5M params, 20K vocab), which is architecturally incompatible with finetune_low_config.json.'
        )
    else:
        raise RuntimeError(
            f'PRETRAINED_CKPT={PRETRAINED_CKPT} does not exist, and no checkpoint_best.pt was found '
            f'anywhere under {search_root}. Upload telugu/model/outputs/submission/checkpoint_best.pt '
            "as a Kaggle dataset (e.g. 'checkpoint-telugu-low') and attach it as an input to this "
            'notebook, or update PRETRAINED_CKPT above to wherever it is actually mounted. This must '
            'be the SUBMISSION checkpoint (6 layers, 10K vocab, ~88MB) -- the normal checkpoint-telugu '
            'dataset (25.5M params, 20K vocab) is architecturally incompatible with finetune_low_config.json.'
        )

# Setup path to import from bundle
sys.path.insert(0, ROOT_DIR)
os.chdir('/kaggle/working')  # For checkpoint/log output

# Install tokenizers if needed
import subprocess
subprocess.run(['pip', 'install', 'tokenizers'], capture_output=True)

print(f'Root: {ROOT_DIR}')
print(f'Pretrained checkpoint (SUBMISSION/low-parameter): {PRETRAINED_CKPT}')
print(f'Output: {OUT_DIR}')
print(f'Finetune config: {FINETUNE_CONFIG_NAME}')
print(f'Resume: {CHECK_DIR}')

Root: /kaggle/input/datasets/kspsvln/lma-telugu-phase2
Pretrained checkpoint (SUBMISSION/low-parameter): /kaggle/input/datasets/kspsvln/checkpoint-telugu/submission_phase-2/checkpoint_best.pt
Output: /kaggle/working/finetune_checkpoints_low
Finetune config: finetune_low_config.json
Resume: None


In [3]:
# Import the real finetuning code (no reimplementation) -- same finetune.py as the normal run,
# now with the added support for a configurable finetune-config filename + architecture/tokenizer
# overrides (see finetune_low_config.json's model_architecture / tokenizer_filename fields).
from finetune.finetune import run_finetuning

print('✅ Imported finetuning code from bundle')

✅ Imported finetuning code from bundle


In [4]:
# Build command-line arguments by mimicking finetune.py's argparse
# Filter out None hyperparams (use finetune_low_config.json defaults)
args_dict = {k: v for k, v in hp.items() if v is not None}

# Handle resume: compute resume_from path (resuming a FINETUNE session, separate from PRETRAINED_CKPT)
resume_from = None
if CHECK_DIR and Path(CHECK_DIR).exists():
    potential_ckpt = Path(CHECK_DIR) / 'checkpoint_last.pt'
    if potential_ckpt.exists():
        resume_from = str(potential_ckpt)
        print(f'Will resume finetuning from: {resume_from}')

# Create args namespace (fields must match finetune.py's run_finetuning() override_config exactly)
args = argparse.Namespace(
    batch_size=args_dict.get('batch_size', None),
    learning_rate=args_dict.get('learning_rate', None),
    num_epochs=args_dict.get('num_epochs', None),
    warmup_steps=args_dict.get('warmup_steps', None),
    weight_decay=args_dict.get('weight_decay', None),
    amp=args_dict.get('amp', True),
    pretrained_ckpt=PRETRAINED_CKPT,
    resume_from=resume_from,
)

print('✅ Arguments prepared')

✅ Arguments prepared


In [5]:
# Run finetuning with the real Trainer subclass (checkpointing, logging, AMP, masked QA loss,
# class-weighted loss, etc. all included) -- finetune_config_name selects finetune_low_config.json
# instead of the default, which is what makes this the low-parameter ablation run.
print('\n' + '='*60)
print(f'Starting {LANGUAGE.upper()} LOW-PARAMETER reasoning finetuning...')
print('='*60 + '\n')

run_finetuning(args, root_dir=ROOT_DIR, data_dir=None, output_dir=OUT_DIR, finetune_config_name=FINETUNE_CONFIG_NAME)

print('\n' + '='*60)
print('✅ Low-parameter finetuning complete!')
print('='*60)


Starting TELUGU LOW-PARAMETER reasoning finetuning...

Device: cuda

Finetuning config:
  language: telugu
  training_phase: reasoning_finetuning_low_parameter_ablation
  batch_size: 4
  learning_rate: 5e-06
  weight_decay: 0.01
  num_epochs: 20
  warmup_steps: 50
  optimizer: adamw
  scheduler: cosine_with_warmup
  loss_function: cross_entropy
  max_grad_norm: 1.0
  seed: 42
  device: auto
  num_workers: 2
  amp: True
  model_architecture: {'vocab_size': 10000, 'd_model': 256, 'num_layers': 6, 'num_heads': 8, 'd_ff': 1024, 'max_seq_len': 128}
  tokenizer_filename: full_wordPiece_level/telugu_wp_tokenizer_low.json
  tokenizer_config_filename: tokenizer_config_low.json
  description: Low-parameter ablation: finetunes the Phase 2 SUBMISSION checkpoint (6 layers, 10K WordPiece vocab, 7.34M params, PPL 881.9 -- telugu/model/outputs/submission/checkpoint_best.pt) instead of the larger, currently-in-progress 25.5M-param/20K-vocab checkpoint that finetune_config.json targets. Same LR/epochs/

Training:  50%|█████     | 2001/4000 [01:28<01:52, 17.70it/s, loss=5.58]

  Saving checkpoint to /kaggle/working/finetune_checkpoints_low/checkpoint_last.pt


Training: 100%|█████████▉| 3999/4000 [02:55<00:00, 20.21it/s, loss=4.15]

  Saving checkpoint to /kaggle/working/finetune_checkpoints_low/checkpoint_last.pt


...

Training:  50%|████▉     | 1998/4000 [01:29<01:25, 23.44it/s, loss=3.24]

  Saving checkpoint to /kaggle/working/finetune_checkpoints_low/checkpoint_last.pt


Training: 100%|█████████▉| 3999/4000 [02:56<00:00, 23.66it/s, loss=3.04]

  Saving checkpoint to /kaggle/working/finetune_checkpoints_low/checkpoint_last.pt


Epoch 20: train_loss=2.1949 val_loss=6.9164 val_ppl=1008.67
  Saving checkpoint to /kaggle/working/finetune_checkpoints_low/checkpoint_last.pt

✓ Finetuning complete!
  Best val_loss: 6.1728
  Best val_ppl: 479.53
✓ Loaded checkpoint_best.pt (step 8000) for final test evaluation

✓ Finetuned test exact-match accuracy (best checkpoint): 0.0875
✓ Saved summary to /kaggle/working/finetune_checkpoints_low/finetune_summary.json

✅ Low-parameter finetuning complete!


In [6]:
# Final summary
import glob
import json

checkpoints = sorted(glob.glob(f'{OUT_DIR}/checkpoint_*.pt'))
logs = sorted(glob.glob(f'{OUT_DIR}/*.log'))
summary_path = Path(OUT_DIR) / 'finetune_summary.json'

print(f'\n📊 Low-parameter finetuning outputs:')
if checkpoints:
    print(f'  Checkpoints:')
    for ckpt in checkpoints:
        size_mb = Path(ckpt).stat().st_size / 1e6
        print(f'    {ckpt} ({size_mb:.1f} MB)')

if logs:
    print(f'  Logs:')
    for log in logs:
        print(f'    {log}')

if summary_path.exists():
    summary = json.load(open(summary_path))
    print(f'\n📈 Pretrained vs. finetuned, LOW-PARAMETER model (compare against finetune_checkpoints/finetune_summary.json for the high-parameter run):')
    print(f'  Pretrained test exact-match accuracy: {summary["pretrained_test_accuracy"]}')
    print(f'  Finetuned test exact-match accuracy:  {summary["finetuned_test_accuracy"]}')
    print(f'  Best val_loss / val_ppl: {summary["best_val_loss"]:.4f} / {summary["best_val_ppl"]:.2f}')

print(f'\n📝 To resume this LOW-parameter finetuning in a later session:')
print(f'  1. Save this notebook output as a Kaggle dataset')
print(f'  2. Set CHECK_DIR = "/kaggle/input/<this-output-dataset>/finetune_checkpoints_low"')
print(f'  3. Run the notebook again')


📊 Low-parameter finetuning outputs:
  Checkpoints:
    /kaggle/working/finetune_checkpoints_low/checkpoint_best.pt (88.2 MB)
    /kaggle/working/finetune_checkpoints_low/checkpoint_last.pt (88.2 MB)
  Logs:
    /kaggle/working/finetune_checkpoints_low/training_telugu.log

📈 Pretrained vs. finetuned, LOW-PARAMETER model (compare against finetune_checkpoints/finetune_summary.json for the high-parameter run):
  Pretrained test exact-match accuracy: 0.0
  Finetuned test exact-match accuracy:  0.0875
  Best val_loss / val_ppl: 6.1728 / 479.53

📝 To resume this LOW-parameter finetuning in a later session:
  1. Save this notebook output as a Kaggle dataset
  2. Set CHECK_DIR = "/kaggle/input/<this-output-dataset>/finetune_checkpoints_low"
  3. Run the notebook again


## Final Test-Set Evaluation (Best FINETUNING Checkpoint, low-parameter model)

Same idea as the normal notebook's final cell, adapted for this model's architecture: reloads `OUT_DIR/checkpoint_best.pt` (the low-parameter run's own best-val-loss checkpoint -- not the high-parameter run's, and not `PRETRAINED_CKPT`) and reports accuracy broken down by question type and held-out vs. seen template phrasing, for direct comparison against the normal notebook's breakdown in the ablation writeup.

In [7]:
import json
import re
from collections import defaultdict

import torch

from finetune.finetune import evaluate_exact_match
from model.transformer import TeluguTransformer
from tokenizer.tokenizer_wrapper import TeluguTokenizer

# Architecture/tokenizer must match finetune_low_config.json exactly (read it directly rather
# than hardcoding, so this cell can't silently drift out of sync with what was actually trained).
with open(Path(ROOT_DIR) / 'configs' / FINETUNE_CONFIG_NAME) as f:
    low_cfg = json.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = TeluguTokenizer(
    tokenizer_path=str(Path(ROOT_DIR) / 'tokenizer' / low_cfg['tokenizer_filename']),
    config_path=str(Path(ROOT_DIR) / 'configs' / low_cfg['tokenizer_config_filename']),
)
model = TeluguTransformer(**low_cfg['model_architecture']).to(device)

best_finetune_ckpt_path = Path(OUT_DIR) / 'checkpoint_best.pt'
assert best_finetune_ckpt_path.exists(), f'{best_finetune_ckpt_path} not found -- run the finetuning cell above first'
ckpt = torch.load(best_finetune_ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"✅ Loaded LOW-PARAMETER FINETUNING best checkpoint from {best_finetune_ckpt_path} (step {ckpt['step']}, epoch {ckpt['epoch']})")

test_path = Path(ROOT_DIR) / 'finetune' / 'data' / 'test.jsonl'
overall_acc = evaluate_exact_match(model, tokenizer, str(test_path), device)
print(f"\n📊 Test exact-match accuracy (low-parameter finetuning-best checkpoint): {overall_acc:.4f}")

def base_pattern(template_id):
    return re.sub(r'_(age|height|weight|price)(_ho)?$', '', template_id)

examples = [json.loads(l) for l in open(test_path, encoding='utf-8') if l.strip()]
correct_by_group, total_by_group = defaultdict(int), defaultdict(int)
correct_ho = total_ho = correct_seen = total_seen = 0
mistakes = []

with torch.no_grad():
    for ex in examples:
        ids = tokenizer.encode(ex['prompt'], add_special_tokens=False)
        if not ids:
            continue
        input_ids = torch.tensor([ids], device=device)
        generated = model.generate(input_ids, max_new_tokens=10, temperature=1.0, greedy=True)
        gen_ids = generated[0].cpu().numpy().tolist()[len(ids):]
        pred_words = tokenizer.decode(gen_ids).strip().split()
        pred = pred_words[0] if pred_words else ''
        gold = ex['answer'].strip()
        is_correct = pred == gold

        group = base_pattern(ex['template_id'])
        correct_by_group[group] += int(is_correct)
        total_by_group[group] += 1

        if ex['template_id'].endswith('_ho'):
            correct_ho += int(is_correct); total_ho += 1
        else:
            correct_seen += int(is_correct); total_seen += 1

        if not is_correct and len(mistakes) < 10:
            mistakes.append({'prompt': ex['prompt'], 'gold': gold, 'pred': pred})

print(f"\nHeld-out-template phrasing vs seen-template phrasing:")
if total_ho:
    print(f"  Held-out (_ho): {correct_ho}/{total_ho} = {correct_ho/total_ho:.4f}")
if total_seen:
    print(f"  Seen:           {correct_seen}/{total_seen} = {correct_seen/total_seen:.4f}")

print(f"\nBy question type:")
for group in sorted(total_by_group):
    c, t = correct_by_group[group], total_by_group[group]
    print(f"  {group:28s} {c:4d}/{t:4d} = {c/t:.4f}")

print(f"\nSample mistakes (up to 10):")
for m in mistakes:
    print(f"  {m['prompt'][:90]}")
    print(f"    gold={m['gold']!r}  pred={m['pred']!r}")

breakdown = {
    'checkpoint_used': str(best_finetune_ckpt_path),
    'model_variant': 'low_parameter',
    'overall_accuracy': overall_acc,
    'held_out_accuracy': correct_ho / total_ho if total_ho else None,
    'seen_template_accuracy': correct_seen / total_seen if total_seen else None,
    'by_question_type': {g: correct_by_group[g] / total_by_group[g] for g in total_by_group},
}
breakdown_path = Path(OUT_DIR) / 'test_eval_breakdown.json'
with open(breakdown_path, 'w', encoding='utf-8') as f:
    json.dump(breakdown, f, indent=2, ensure_ascii=False)
print(f"\n✅ Saved breakdown to {breakdown_path}")

✅ Loaded LOW-PARAMETER FINETUNING best checkpoint from /kaggle/working/finetune_checkpoints_low/checkpoint_best.pt (step 8000, epoch 2)

📊 Test exact-match accuracy (low-parameter finetuning-best checkpoint): 0.0875

Held-out-template phrasing vs seen-template phrasing:
  Held-out (_ho): 42/488 = 0.0861
  Seen:           123/1512 = 0.0813

By question type:
  equal                           0/ 105 = 0.0000
  pairwise_value                  0/ 619 = 0.0000
  pairwise_yesno                165/ 504 = 0.3274
  three_superlative_max           0/ 178 = 0.0000
  three_superlative_min           0/ 187 = 0.0000
  transitive_max                  0/ 169 = 0.0000
  transitive_min                  0/ 149 = 0.0000
  transitive_yesno                0/  89 = 0.0000

Sample mistakes (up to 10):
  ప్రశ్న: శ్రీను, వెంకటేశ్ కంటే ఎక్కువ వయస్సు కలిగి ఉన్నారు. వెంకటేశ్, వెంకటేశ్వరరావు కంటే ఎ
    gold='అవును'  pred='కాదు'
  ప్రశ్న: వెంకటేశ్ యొక్క వయస్సు 5 సంవత్సరాలు, మరియు రవి యొక్క వయస్సు 15 సంవత్సరాలు. వీటి